[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abu-dhabi-ai-proptech-challenge/starter-kit/blob/main/notebooks/explore_sample_data.ipynb)

# Abu Dhabi AI PropTech Challenge — Sample Data Walkthrough

**Building the Intelligence Layer for Land, Investment and Communities**

This notebook loads the four synthetic datasets, shows what's in them, and demonstrates the cross-dataset joins that power most prototype ideas.

> All data is **synthetic** — generated for this challenge. Values are plausible but not real.

In [ ]:
import pandas as pd
from pathlib import Path

# Works anywhere: local clone uses ../data, Colab/standalone pulls from Hugging Face
DATA = "../data" if Path("../data/sample_parcels.csv").exists() else \
    "https://huggingface.co/datasets/eVoost/abu-dhabi-ai-proptech-challenge/resolve/main"

parcels = pd.read_csv(f"{DATA}/sample_parcels.csv")
investors = pd.read_csv(f"{DATA}/sample_investors.csv")
transactions = pd.read_csv(f"{DATA}/sample_transactions.csv", parse_dates=["date"])
communities = pd.read_csv(f"{DATA}/sample_communities.csv")

for name, df in {"parcels": parcels, "investors": investors, "transactions": transactions, "communities": communities}.items():
    print(f"{name:<13} {df.shape[0]} rows x {df.shape[1]} cols")

## 1. Land: where is the development potential?

In [ ]:
parcels.head()

In [ ]:
# Vacant parcels ranked by development potential
vacant = parcels[parcels["current_status"] == "vacant"]
vacant.nlargest(8, "development_potential_score")[
    ["parcel_id", "district", "land_use", "parcel_size_sqm",
     "development_potential_score", "infrastructure_score", "recommended_use"]
]

In [ ]:
# Average potential vs infrastructure by district — a simple opportunity map
parcels.groupby("district")[["development_potential_score", "infrastructure_score", "estimated_value_aed"]].mean().round(0).sort_values("development_potential_score", ascending=False)

## 2. Market: what do transactions say?

In [ ]:
# Price per sqm by district and asset type
transactions.pivot_table(index="district", columns="asset_type", values="price_per_sqm", aggfunc="mean").round(0)

In [ ]:
# Transaction volume over time
transactions.set_index("date").resample("ME")["transaction_value_aed"].sum().plot(
    kind="bar", title="Monthly transaction volume (AED)", figsize=(10, 4)
);

## 3. Communities: where is demand unmet?

In [ ]:
# Districts under service pressure, and what would help
communities.sort_values("service_demand_index", ascending=False)[
    ["district", "population_estimate", "service_demand_index",
     "mobility_score", "resident_experience_score", "optimization_opportunity"]
].head(8)

## 4. The join that powers cross-cutting prototypes

All four datasets share `district`. Joining land supply against community demand is the core of many Decision Intelligence ideas.

In [ ]:
demand = communities.groupby("district")["service_demand_index"].mean().rename("avg_service_demand")
supply = parcels[parcels["current_status"] == "vacant"].groupby("district")["parcel_size_sqm"].sum().rename("vacant_sqm")
prices = transactions.groupby("district")["price_per_sqm"].mean().rename("avg_price_sqm")

opportunity = pd.concat([demand, supply, prices], axis=1).fillna(0).round(0)
opportunity["signal"] = (opportunity["avg_service_demand"] > 70) & (opportunity["vacant_sqm"] > 10000)
opportunity.sort_values("avg_service_demand", ascending=False)

Districts where `signal` is `True` have both high unmet demand **and** available land — exactly the kind of insight a prototype can surface, explain, and act on.

## Where to go from here

- Run the example agents in [`../examples/`](../examples/) — score + explain, match + rank, Q&A patterns
- Wrap your analysis in the `project-template` dashboard for a demo-ready UI
- Connect an LLM where the examples mark `# --- CONNECT YOUR MODEL HERE ---`